# ASTE Model v1 — (Aspect, Opinion, VA) Triplet Extraction

Fine-tunes **T5-small** to extract `(Aspect, Opinion, Valence#Arousal)` triplets from review text.

- **Datasets**: Restaurant (REST) + Laptop (LAP)
- **Category is ignored** (only Aspect, Opinion, VA are used)
- **Output**: JSONL with `ID` and `Triplet` fields

**Improvements in this version:**
1. **Text preprocessing** — normalize whitespace before feeding to model
2. **Cosine LR scheduler** with warmup ratio for smoother convergence
3. **Gradient accumulation** — effective batch size of 32 (4 × 8) for more stable gradients
4. **Label smoothing** (0.1) — prevents overconfident CE loss
5. **Better beam search** — 5 beams + `no_repeat_ngram_size=3` + `repetition_penalty=1.2`
6. **tqdm progress bar** during inference

In [ ]:
#!pip install transformers datasets sentencepiece protobuf accelerate -q

In [29]:
import json
import re
import os
import random
import numpy as np
import torch
from pathlib import Path
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import (
    T5ForConditionalGeneration,
    T5Tokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
PyTorch: 2.10.0+cu128
GPU: Tesla T4


In [30]:
import sys
import __main__

# This line tricks the transformers library into thinking it's running from a file
if not hasattr(__main__, "__file__"):
    __main__.__file__ = "run_aste.py"

# Now your initialization will work
model = T5WithLSTM.from_pretrained(MODEL_NAME)

FileNotFoundError: [Errno 2] No such file or directory: 'kaggle_notebook.py'

## 1. Configuration

In [20]:
# ── Paths (Kaggle) ──
DATA_DIR    = Path("/kaggle/input/datasets/chaitanyajx1/datasetnlp")
REST_FILE   = DATA_DIR / "restaurant_train.jsonl"
LAP_FILE    = DATA_DIR / "laptop_train.jsonl"
MODEL_DIR   = Path("/kaggle/working/aste_t5small_model")
OUTPUT_FILE = Path("/kaggle/working/predictions.jsonl")

# ── Hyper-parameters (Improved) ──
MODEL_NAME           = "t5-small"              # 60M params
MAX_INPUT_LEN        = 256
MAX_TARGET_LEN       = 256
BATCH_SIZE           = 8
GRAD_ACCUM_STEPS     = 4                       # effective batch = 8 × 4 = 32
LEARNING_RATE        = 3e-4
NUM_EPOCHS           = 15                      # more epochs with early stopping
WARMUP_RATIO         = 0.1                     # 10% of steps for warmup
VAL_SPLIT            = 0.1
LOGGING_STEPS        = 100
NUM_BEAMS            = 5                       # more beams than default 4
LENGTH_PENALTY       = 1.0
LABEL_SMOOTHING      = 0.1                     # prevents overconfident predictions
NO_REPEAT_NGRAM_SIZE = 3                       # stop repeated triplet phrases
REPETITION_PENALTY   = 1.2                     # further penalise duplicate tokens

## 2. Load & Preprocess Data

In [21]:
def preprocess_text(text):
    """Normalize whitespace for cleaner input."""
    text = re.sub(r'[\t\n\r\f\v]+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def load_jsonl(filepath):
    """Load JSONL file -> list of dicts."""
    data = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                data.append(json.loads(line))
    return data


rest_data = load_jsonl(REST_FILE)
lap_data  = load_jsonl(LAP_FILE)
all_data  = rest_data + lap_data

print(f"Restaurant (REST): {len(rest_data)}")
print(f"Laptop     (LAP):  {len(lap_data)}")
print(f"Total:             {len(all_data)}")

Restaurant (REST): 2284
Laptop     (LAP):  4076
Total:             6360


### 2.1 Linearize Triplets for Seq2Seq

- **Input**: `"extract triplets: <preprocessed review text>"`
- **Target**: `"( Aspect | Opinion | V#A ) ; ( Aspect | Opinion | V#A ) ; ..."`

Category is **ignored**.

In [22]:
def record_to_seq2seq(record):
    """
    Convert a single JSONL record to (input_text, target_text) for T5.
    Now includes text preprocessing. Category is ignored.
    """
    text = preprocess_text(record["Text"])
    input_text = f"extract triplets: {text}"

    triplet_strs = []
    for quad in record["Quadruplet"]:
        aspect  = quad.get("Aspect", "NULL")
        opinion = quad.get("Opinion", "NULL")
        va      = quad.get("VA", "5.00#5.00")
        triplet_strs.append(f"( {aspect} | {opinion} | {va} )")

    target_text = " ; ".join(triplet_strs)
    return input_text, target_text


# Build pairs
pairs = []
for rec in all_data:
    inp, tgt = record_to_seq2seq(rec)
    pairs.append({"id": rec["ID"], "input": inp, "target": tgt})

print(f"Total seq2seq pairs: {len(pairs)}")
print()
print("Example:")
print(f"  INPUT:  {pairs[0]['input'][:120]}...")
print(f"  TARGET: {pairs[0]['target'][:120]}...")

Total seq2seq pairs: 6360

Example:
  INPUT:  extract triplets: ca n ' t wait wait for my next visit ....
  TARGET: ( NULL | NULL | 6.75#6.38 )...


### 2.2 Train / Validation Split

In [23]:
random.shuffle(pairs)
split_idx = int(len(pairs) * (1 - VAL_SPLIT))
train_pairs = pairs[:split_idx]
val_pairs   = pairs[split_idx:]

print(f"Train: {len(train_pairs)}")
print(f"Val:   {len(val_pairs)}")

Train: 5724
Val:   636


## 3. Tokenization & Dataset

In [24]:
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME, legacy=True)

class ASTEDataset(Dataset):
    def __init__(self, pairs, tokenizer, max_input_len, max_target_len):
        self.pairs = pairs
        self.tokenizer = tokenizer
        self.max_input_len = max_input_len
        self.max_target_len = max_target_len

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        pair = self.pairs[idx]
        input_enc = self.tokenizer(
            pair["input"],
            max_length=self.max_input_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        target_enc = self.tokenizer(
            pair["target"],
            max_length=self.max_target_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        labels = target_enc["input_ids"].squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            "input_ids":      input_enc["input_ids"].squeeze(),
            "attention_mask": input_enc["attention_mask"].squeeze(),
            "labels":         labels,
        }

train_dataset = ASTEDataset(train_pairs, tokenizer, MAX_INPUT_LEN, MAX_TARGET_LEN)
val_dataset   = ASTEDataset(val_pairs,   tokenizer, MAX_INPUT_LEN, MAX_TARGET_LEN)

print(f"Train samples: {len(train_dataset)}")
print(f"Val   samples: {len(val_dataset)}")

sample = train_dataset[0]
print(f"input_ids shape:  {sample['input_ids'].shape}")
print(f"labels shape:     {sample['labels'].shape}")

Train samples: 5724
Val   samples: 636
input_ids shape:  torch.Size([256])
labels shape:     torch.Size([256])


## 4. Model & Training

In [25]:
import torch.nn as nn
from transformers import T5ForConditionalGeneration

class T5WithLSTM(T5ForConditionalGeneration):
    def __init__(self, config):
        super().__init__(config)
        # Inject a BiLSTM layer on top of the encoder
        self.lstm = nn.LSTM(config.d_model, config.d_model // 2, batch_first=True, bidirectional=True)
        # Projection to ensure the expected hidden dimension for the cross-attention wrapper
        self.lstm_proj = nn.Linear(config.d_model, config.d_model)
        
    def prepare_inputs_for_generation(self, *args, **kwargs):
        return super().prepare_inputs_for_generation(*args, **kwargs)
        
    def forward(self, input_ids=None, attention_mask=None, decoder_input_ids=None, decoder_attention_mask=None, head_mask=None, decoder_head_mask=None, cross_attn_head_mask=None, encoder_outputs=None, past_key_values=None, inputs_embeds=None, decoder_inputs_embeds=None, labels=None, use_cache=None, output_attentions=None, output_hidden_states=None, return_dict=None):
        if encoder_outputs is None:
            # Pass through the encoder
            encoder_outputs = self.encoder(
                input_ids=input_ids,
                attention_mask=attention_mask,
                inputs_embeds=inputs_embeds,
                head_mask=head_mask,
                output_attentions=output_attentions,
                output_hidden_states=output_hidden_states,
                return_dict=return_dict,
            )
            
        # Unpack the encoder outputs
        if isinstance(encoder_outputs, tuple):
            hidden_states = encoder_outputs[0]
        else:
            hidden_states = encoder_outputs.last_hidden_state
            
        # Pass encoder hidden states through BiLSTM
        lstm_out, _ = self.lstm(hidden_states)
        hidden_states = self.lstm_proj(lstm_out)
        
        # Wrap the new hidden states back into the tuple wrapper
        if isinstance(encoder_outputs, tuple):
            encoder_outputs = (hidden_states,) + encoder_outputs[1:]
        else:
            encoder_outputs.last_hidden_state = hidden_states
            
        return super().forward(
            attention_mask=attention_mask,
            decoder_input_ids=decoder_input_ids,
            decoder_attention_mask=decoder_attention_mask,
            head_mask=head_mask,
            decoder_head_mask=decoder_head_mask,
            cross_attn_head_mask=cross_attn_head_mask,
            encoder_outputs=encoder_outputs,
            past_key_values=past_key_values,
            inputs_embeds=inputs_embeds,
            decoder_inputs_embeds=decoder_inputs_embeds,
            labels=labels,
            use_cache=use_cache,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )

# Initialise custom model
model = T5WithLSTM.from_pretrained(MODEL_NAME)
print(f"Model: T5WithLSTM({MODEL_NAME})")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")


AttributeError: module '__main__' has no attribute '__file__'

In [18]:
training_args = Seq2SeqTrainingArguments(
    output_dir=str(MODEL_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,   # effective batch = 32
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_ratio=WARMUP_RATIO,                       # 10% warmup (was fixed 200 steps)
    lr_scheduler_type="cosine",                       # cosine annealing (was linear)
    label_smoothing_factor=LABEL_SMOOTHING,           # prevents overconfident predictions
    logging_steps=LOGGING_STEPS,
    save_strategy="epoch",
    save_total_limit=2,
    eval_strategy="epoch",
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,
    fp16=torch.cuda.is_available(),
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    seed=SEED,
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
)

total_steps = (len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM_STEPS)) * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

print("Trainer ready.")
print(f"  Model:           {MODEL_NAME} ({sum(p.numel() for p in model.parameters()):,} params)")
print(f"  Device:          {DEVICE}")
print(f"  FP16:            {training_args.fp16}")
print(f"  Epochs:          {NUM_EPOCHS}")
print(f"  Batch size:      {BATCH_SIZE} (effective: {BATCH_SIZE * GRAD_ACCUM_STEPS})")
print(f"  LR:              {LEARNING_RATE} (cosine annealing)")
print(f"  Label smoothing: {LABEL_SMOOTHING}")
print(f"  Warmup:          {warmup_steps} steps ({WARMUP_RATIO*100:.0f}%)")
print(f"  Total steps:     {total_steps}")
print(f"  Train samples:   {len(train_dataset)}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


NameError: name 'model' is not defined

In [ ]:
# ══════════════════════════════════════════════════════
#  TRAINING  (Tesla T4 GPU: ~20-40 min)
# ══════════════════════════════════════════════════════
trainer.train()
print("\n✅ Training complete!")

/usr/local/lib/python3.12/dist-packages/transformers/data/data_collator.py:600: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.int64)


Epoch,Training Loss,Validation Loss
1,No log,5.002709
2,33.878877,4.302619
3,18.611283,4.099024
4,16.976381,3.991191
5,16.310721,3.941084
6,15.887959,3.906767
7,15.578029,3.885463
8,15.397738,3.863149
9,15.248997,3.867418
10,15.070266,3.847462


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].



✅ Training complete!


In [ ]:
# Save the best model
trainer.save_model(str(MODEL_DIR / "best"))
tokenizer.save_pretrained(str(MODEL_DIR / "best"))
print(f"Model saved to: {MODEL_DIR / 'best'}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: /kaggle/working/aste_t5small_model/best


## 5. Inference — Parse Generated Triplets

In [ ]:
def parse_triplets(generated_text):
    """
    Parse linearized output back into structured triplets.
    Expected format: ( Aspect | Opinion | V#A ) ; ...
    """
    triplets = []
    pattern = r"\(\s*(.+?)\s*\|\s*(.+?)\s*\|\s*([\d.]+#[\d.]+)\s*\)"
    for match in re.finditer(pattern, generated_text):
        aspect  = match.group(1).strip()
        opinion = match.group(2).strip()
        va      = match.group(3).strip()

        # Clamp VA to valid range [1.00, 9.00]
        try:
            v, a = va.split("#")
            v, a = float(v), float(a)
            v = max(1.0, min(9.0, v))
            a = max(1.0, min(9.0, a))
            va = f"{v:.2f}#{a:.2f}"
        except:
            va = "5.00#5.00"

        triplets.append({
            "Aspect":  aspect,
            "Opinion": opinion,
            "VA":      va,
        })
    return triplets

# Quick test
test_str = "( Indian food | average to good | 6.75#6.38 ) ; ( delivery | terrible | 2.88#6.62 )"
print("Parse test:", parse_triplets(test_str))

Parse test: [{'Aspect': 'Indian food', 'Opinion': 'average to good', 'VA': '6.75#6.38'}, {'Aspect': 'delivery', 'Opinion': 'terrible', 'VA': '2.88#6.62'}]


In [ ]:
def predict_triplets(texts, model, tokenizer, max_len=256, batch_size=16,
                     num_beams=5, length_penalty=1.0,
                     no_repeat_ngram_size=3, repetition_penalty=1.2):
    """
    Generate triplet predictions with improved beam search.
    - no_repeat_ngram_size: stops model looping the same triplet phrase
    - repetition_penalty:   further penalises duplicate tokens in output
    """
    model.eval()
    all_triplets = []
    batches = list(range(0, len(texts), batch_size))

    for i in tqdm(batches, desc="Generating", unit="batch"):
        batch_texts = texts[i : i + batch_size]
        inputs = [f"extract triplets: {preprocess_text(t)}" for t in batch_texts]

        encoded = tokenizer(
            inputs,
            max_length=max_len,
            padding=True,
            truncation=True,
            return_tensors="pt",
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **encoded,
                max_length=max_len,
                num_beams=num_beams,
                length_penalty=length_penalty,
                no_repeat_ngram_size=no_repeat_ngram_size,  # stops repetitive phrases
                repetition_penalty=repetition_penalty,       # penalises duplicate tokens
                early_stopping=True,
            )

        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

        for gen_text in decoded:
            triplets = parse_triplets(gen_text)
            all_triplets.append(triplets)

    return all_triplets

print("Inference function ready.")

Inference function ready.


## 6. Evaluate on Validation Set

In [ ]:
# Get validation texts and ground-truth
val_texts  = [p["input"].replace("extract triplets: ", "") for p in val_pairs]
val_golds  = [p["target"] for p in val_pairs]
val_ids    = [p["id"] for p in val_pairs]

# Predict
val_preds = predict_triplets(
    val_texts, model, tokenizer,
    max_len=MAX_TARGET_LEN,
    num_beams=NUM_BEAMS,
    length_penalty=LENGTH_PENALTY,
    no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
    repetition_penalty=REPETITION_PENALTY,
)

# Show some examples
for i in range(min(8, len(val_preds))):
    print(f"\n{'='*70}")
    print(f"ID:    {val_ids[i]}")
    print(f"Text:  {val_texts[i][:100]}...")
    print(f"Gold:  {val_golds[i][:100]}...")
    print(f"Pred:  {val_preds[i]}")

Generating:   0%|          | 0/40 [00:00<?, ?batch/s]


ID:    rest16_quad_train_120
Text:  this is the best sushi in new york city - hands down ....
Gold:  ( sushi | best | 7.00#6.83 )...
Pred:  [{'Aspect': 'sushi', 'Opinion': 'best', 'VA': '8.00#8.00'}]

ID:    rest16_quad_dev_46
Text:  check out the secret back room ....
Gold:  ( back room | secret | 6.38#6.50 )...
Pred:  [{'Aspect': 'back room', 'Opinion': 'NULL', 'VA': '5.00#5.00'}]

ID:    laptop_quad_train_2208
Text:  barely have it for 6 months and everything ' s going haywire ....
Gold:  ( NULL | haywire | 3.33#6.67 )...
Pred:  [{'Aspect': 'NULL', 'Opinion': 'haywire', 'VA': '3.50#6.50'}]

ID:    laptop_quad_train_1949
Text:  to me back - lighting on a keyboard is a make or break so i might return the laptop though ( bought ...
Gold:  ( laptop | NULL | 4.62#5.12 )...
Pred:  [{'Aspect': 'keyboard', 'Opinion': 'NULL', 'VA': '4.50#5.17'}]

ID:    rest16_quad_train_455
Text:  will not be back ....
Gold:  ( NULL | NULL | 4.12#4.62 )...
Pred:  [{'Aspect': 'NULL', 'Opinion': 'not be back

In [ ]:
# ── Simple metrics ──
def compute_metrics(val_pairs, val_preds):
    """Compute aspect/opinion extraction F1 and VA MAE."""
    total_gold = 0
    total_pred = 0
    correct    = 0
    va_errors  = []

    for pair, preds in zip(val_pairs, val_preds):
        gold_triplets = parse_triplets(pair["target"])
        total_gold += len(gold_triplets)
        total_pred += len(preds)

        gold_set = {(g["Aspect"].lower(), g["Opinion"].lower()): g["VA"] for g in gold_triplets}
        for p in preds:
            key = (p["Aspect"].lower(), p["Opinion"].lower())
            if key in gold_set:
                correct += 1
                try:
                    gv, ga = gold_set[key].split("#")
                    pv, pa = p["VA"].split("#")
                    va_errors.append(abs(float(gv) - float(pv)) + abs(float(ga) - float(pa)))
                except:
                    pass

    prec = correct / total_pred if total_pred else 0
    rec  = correct / total_gold if total_gold else 0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) else 0
    avg_va = np.mean(va_errors) if va_errors else float('inf')

    print(f"\n{'='*50}")
    print(f"Span Extraction:")
    print(f"  Gold: {total_gold}  Pred: {total_pred}  Correct: {correct}")
    print(f"  P={prec:.4f}  R={rec:.4f}  F1={f1:.4f}")
    print(f"VA Error (matched): {avg_va:.4f}  (n={len(va_errors)})")

compute_metrics(val_pairs, val_preds)


Span Extraction:
  Gold: 920  Pred: 810  Correct: 533
  P=0.6580  R=0.5793  F1=0.6162
VA Error (matched): 1.1718  (n=533)
